# 2. EDA Simplificado

En esta sección se realiza un análisis exploratorio básico del dataset con el objetivo de entender su estructura y preparar el posterior preprocesamiento.

In [6]:
import pandas as pd

df = pd.read_pickle("bank_17.pkl")
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,None,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


## 2.1 Dimensiones del dataset

In [7]:
df.shape

(11000, 17)

El dataset contiene 11000 instancias (filas) y 17 variables (columnas).

## 2.2 Tipos de variables

In [8]:
df.dtypes

age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
deposit      object
dtype: object

In [9]:
num_vars = df.select_dtypes(include=['int64','float64']).columns
cat_vars = df.select_dtypes(include=['object']).columns

num_vars, cat_vars

(Index(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'], dtype='object'),
 Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
        'month', 'poutcome', 'deposit'],
       dtype='object'))

Se identifican 7 variables numéricas:

- age
- balance
- day
- duration
- campaign
- pdays
- previous

Y 10 variables categóricas:

- job
- marital
- education
- default
- housing
- loan
- contact
- month
- poutcome
- deposit

Las variables numéricas son aquellas de tipo entero o flotante, mientras que las categóricas son de tipo objeto.

La variable objetivo es `deposit`, por lo que se trata de un problema de clasificación binaria.

## 2.3 Cardinalidad de variables categóricas

In [10]:
for col in cat_vars:
    print(col, df[col].nunique())

job 12
marital 3
education 4
default 2
housing 2
loan 2
contact 3
month 12
poutcome 4
deposit 2


Se comprueba la cardinalidad de las variables categóricas.

Se observan dos variables con alta cardinalidad (más de 10 valores únicos):

- `job` (12 categorías)
- `month` (12 categorías)

Estas variables deberán tratarse adecuadamente en el preprocesamiento. Aunque 12 categorías no supone una cardinalidad extremadamente alta, el uso de One-Hot Encoding incrementará la dimensionalidad del conjunto de datos.

## 2.4 Valores faltantes

In [11]:
df.isnull().sum()

age            0
job            0
marital        0
education    386
default        0
balance        0
housing        0
loan           0
contact        0
day            0
month          0
duration       0
campaign       0
pdays          0
previous       0
poutcome       0
deposit        0
dtype: int64

In [12]:
missing = df.isnull().sum()
missing[missing > 0]

education    386
dtype: int64

Se observan valores faltantes únicamente en la variable `education`, con un total de 386 valores nulos.

El resto de variables no presentan valores faltantes.

Dado que `education` es una variable categórica, estos valores deberán tratarse durante el preprocesamiento mediante técnicas de imputación apropiadas para variables categóricas, como la imputación por la moda o la creación de una categoría adicional.

## 2.5 Columnas constantes o tipo ID

In [14]:
for col in df.columns:
    if df[col].nunique() == 1:
        print(col)

No se observan columnas constantes ni columnas que actúen como identificadores únicos.

## 2.6 Análisis de la variable objetivo

In [15]:
df['deposit'].value_counts()

deposit
no     5780
yes    5220
Name: count, dtype: int64

In [16]:
df['deposit'].value_counts(normalize=True)

deposit
no     0.525455
yes    0.474545
Name: proportion, dtype: float64

La variable `deposit` indica si el cliente suscribió un depósito.

Se trata de un problema de clasificación binaria, con dos clases: "yes" y "no".

La distribución de clases es:

- 52.55% para la clase "no"
- 47.45% para la clase "yes"

Por tanto, el dataset está ligeramente desbalanceado, aunque la diferencia entre clases no es significativa. No se observa un desbalanceo severo.

## 2.7 Análisis de la variable `pdays`

In [17]:
df['pdays'].value_counts()

pdays
-1      8203
 92      105
 182      88
 91       82
 181      79
        ... 
 651       1
 214       1
 24        1
 701       1
 118       1
Name: count, Length: 472, dtype: int64

In [18]:
(df['pdays'] == -1).mean()

np.float64(0.7457272727272727)

La variable `pdays` representa el número de días transcurridos desde el último contacto en una campaña anterior.

Se observa que el valor -1 aparece 8203 veces, lo que representa aproximadamente el 74.57% del dataset.

Este valor no representa un número real de días, sino una codificación especial que indica ausencia de contacto previo.

Esto indica que una gran parte de los clientes no fueron contactados previamente.

Por tanto, el valor -1 no debe interpretarse como un valor numérico válido, y será necesario tratarlo adecuadamente durante el preprocesamiento.

Una estrategia razonable sería:

- Convertir el valor -1 en un valor nulo (NaN) y posteriormente imputarlo.
- O crear una variable binaria adicional que indique si el cliente fue contactado previamente, manteniendo `pdays` solo para los casos válidos.

Este tratamiento es importante, ya que dejar el valor -1 como un número puede inducir errores en los modelos.

## 2.8 Conclusiones del EDA

El dataset contiene variables numéricas y categóricas, con valores faltantes únicamente en la variable `education`, que deberán imputarse.

Existen variables categóricas con alta cardinalidad (`job` y `month`), que requerirán codificación adecuada.

El problema es de clasificación binaria y el conjunto presenta un ligero desbalanceo entre clases.

La variable `pdays` requiere un tratamiento especial debido a la presencia masiva del valor -1, que representa ausencia de contacto previo.